# Libraries

In [1]:

# Libraries


library(lme4)
library(tidyverse)
library(here)
library(Matrix)

library(flexplot)
library(lmerTest)
library(emmeans)

library(performance)
library(car)


library(parallel)
library(glue)

library(ggplot2)
library(sjPlot)
library(permuco)
library(permuco4brain)

library(coin)

library(future)
library(igraph)

library(lme4)
library(lmerTest)
library(performance)
library(DHARMa)
library(emmeans)
library(dplyr)

Cargando paquete requerido: Matrix

Warning message:
"package 'ggplot2' was built under R version 4.5.3"
Warning message:
"package 'purrr' was built under R version 4.5.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ tidyr::expand() masks Matrix::expand()
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
✖ tidyr::pack()   masks Matrix::pack()
✖ tidyr::unpack() masks Matrix::unpack()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
here() starts at G:/PROYECTO_SELF/CODE_self


Adjuntando el paquete: 'flexplot'


The following object is masked from 'package:ggplot2':

    flip_data


Warning message:
"package 'lm

In [2]:

# Paths
source("G:/PROYECTO_SELF/CODE_SELF_Rstudio/R/get_paths_SELF_R.R")



# Editable parameters
disco <- "g"
modality <- "visual"
layer_script <- "event"
subj <- "sub-V1001"
type_epoch <- "emoc"
# Generate paths
paths <- get_paths_SELF_R(disco, modality, layer_script, subj)

# Additional paths
statistical_models <- "G:/PROYECTO_SELF/models output"

# (Optional) equivalent to globals().update(path_dict)
list2env(paths, envir = knitr::knit_global())

# Display all generated paths
cat("\n📁 Generated paths:\n")
for (nm in names(paths)) {
  cat(sprintf("%-20s → %s\n", nm, paths[[nm]]))
}


✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event/epochs_event

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event/ICA_event

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event/epochs_clean_event

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event/epochs_matlab_event

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event/evoked_event

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/channels_structure

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_source/source_event

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_source/source_event/raw_hsp

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_source/source_event/fwd

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_source/source_event/inverse

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_analysis/analysis_event/acw_event

✅ Folder created: g:/PROYECTO_SELF/SELF_visual/output_analysis/a

<environment: R_GlobalEnv>


📁 Generated paths:
general_datadir      → g:/PROYECTO_SELF
datadir              → g:/PROYECTO_SELF/SELF_visual
BRAIN_VISION_SELF    → g:/PROYECTO_SELF/SELF_visual/BRAIN_VISION_SELF
export_generic_data  → g:/PROYECTO_SELF/SELF_visual/BRAIN_VISION_SELF/export_generic_data
output_preproc       → g:/PROYECTO_SELF/SELF_visual/output_preproc
preproc_path         → g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event
channels_structure_path → g:/PROYECTO_SELF/SELF_visual/channels_structure
epochs_path          → g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event/epochs_event
ICA_path             → g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event/ICA_event
epochs_clean_path    → g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event/epochs_clean_event
epochs_matlab_path   → g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event/epochs_matlab_event
evoked_path          → g:/PROYECTO_SELF/SELF_visual/output_preproc/preproc_event/evoked_event
source_path          → g:/

In [3]:
# ------------------------------------------------------------
# Load adjacency edge list for permuco4brain
# ------------------------------------------------------------


edges_path <- file.path(
  channels_structure_path,
  paste0("adjacency_reduced_edges_", modality, ".csv")
)

edges <- read.csv(edges_path, stringsAsFactors = FALSE)

cat("\n✅ Edge list loaded from:\n")
cat(edges_path, "\n")

cat("N edges:", nrow(edges), "\n")
cat("Columns:", paste(colnames(edges), collapse = ", "), "\n")

# Build igraph object for permuco4brain
graph_channels <- graph_from_data_frame(
  d = edges[, c("from_channel", "to_channel")],
  directed = FALSE
)

cat("✅ graph_channels created\n")
cat("N vertices:", vcount(graph_channels), "\n")
cat("N edges:", ecount(graph_channels), "\n")

# Optional sanity check
print(head(V(graph_channels)$name))


✅ Edge list loaded from:
g:/PROYECTO_SELF/SELF_visual/channels_structure/adjacency_reduced_edges_visual.csv 
N edges: 154 
Columns: from_index, to_index, from_channel, to_channel 
✅ graph_channels created
N vertices: 59 
N edges: 154 
[1] "Fp1" "Fpz" "Fp2" "AF7" "AF3" "AF4"


In [4]:
# Filtering flags 
filtering <- TRUE

if (filtering) {
  lfreq <- 1
  hfreq <- 40
  filter_name <- paste0("filt_", lfreq, "-", hfreq)
  message(sprintf("Filtering applied: %s-%s Hz", lfreq, hfreq))
} else {
  message("No filtering applied.")
}

# -------------------------------
# Crop option
# -------------------------------
crop_epochs <- NULL

if (type_epoch == "emoc") {
  crop_epochs <- NULL
}

# -------------------------------
# Variables of script
# -------------------------------
# Modify this only if you want to use dynamic analysis, or static event analysis
if (layer_script == "event") {
  dynamic <- FALSE
} else if (layer_script == "block") {
  dynamic <- FALSE  # ALWAYS false in block
}

# -------------------------------
# Build FOOOF suffix
# Structure: type_epoch + filter_name + aperiodic_mode + select_channels + layer_script + crop
# -------------------------------
aperiodic_mode <- "fixed"

suffix_fooof <- c(type_epoch)

if (filtering) {
  suffix_fooof <- c(suffix_fooof, filter_name)
}

suffix_fooof <- c(suffix_fooof, aperiodic_mode)

if (exists("select_channels") && !is.null(select_channels) && select_channels != "") {
  suffix_fooof <- c(suffix_fooof, select_channels)
}

suffix_fooof <- c(suffix_fooof, layer_script)

if (!is.null(crop_epochs)) {
  suffix_fooof <- c(suffix_fooof, paste0("crop_", crop_epochs))
}

suffix_fooof_str <- paste(suffix_fooof, collapse = "_")

# -------------------------------
# Path of analysis
# -------------------------------
path <- ACW_path

# -------------------------------
# Table path
# -------------------------------
if (dynamic) {
  table_path <- file.path(
    ACW_path,
    sprintf("table_merged_dynamic_ACW_FOOOF_results_%s.csv", suffix_fooof_str)
  )
} else {
  table_path <- file.path(
    ACW_path,
    sprintf("table_merged_ACW_FOOOF_%s.csv", suffix_fooof_str)
  )
}

# Print table name and full path
table_name <- basename(table_path)
message(sprintf("Table name: %s", table_name))
message(sprintf("Table path: %s", table_path))

maxfun <- 1e5

# Change contrasts so they are valid for interaction and no interaction
options(contrasts = c("contr.sum", "contr.poly"))

# Read merged table
df <- readr::read_csv(table_path, show_col_types = FALSE)

Filtering applied: 1-40 Hz

Table name: table_merged_ACW_FOOOF_emoc_filt_1-40_fixed_event.csv

Table path: g:/PROYECTO_SELF/SELF_visual/output_analysis/analysis_event/acw_event/table_merged_ACW_FOOOF_emoc_filt_1-40_fixed_event.csv



In [ ]:
# Names of all columns
colnames(df)

# Factorization and other transformations

In [5]:
df <- df %>%
  rename(
    ACW_50 = acw_50_elect_all_epoch_all,
    ACW_0  = acw_0_elect_all_epoch_all
  )

if (layer_script == "event") {
  
  df <- df %>%
    mutate(
      Subject = factor(Subject),
      Condition_self = factor(Condition_self),
      Condition_emotion = factor(Condition_emotion),
      Condition_gaze = factor(Condition_gaze),
      Channel = factor(Channel),
    )
  
} 


dependent_variables <- c("ACW_50", "ACW_0")  # modify as needed


# 1. permutations for emoc epochs

### Average at subject level 

In [6]:


nuisance_vars <- c("delta", "theta", "alpha", "beta", "gamma")

# 1) Promediar a nivel Subject × Condition
df_subj <- df %>%
  mutate(
    Subject = factor(Subject),
    Condition_self = factor(Condition_self),
    Condition_emotion = factor(Condition_emotion)

  ) %>%
  group_by(Subject, Condition_self, Condition_emotion) %>%
  summarise(
    ACW_50 = mean(ACW_50, na.rm = TRUE),
    ACW_0 = mean(ACW_0, na.rm = TRUE),
    delta = mean(delta, na.rm = TRUE),
    theta = mean(theta, na.rm = TRUE),
    alpha = mean(alpha, na.rm = TRUE),
    beta  = mean(beta,  na.rm = TRUE),
    gamma = mean(gamma, na.rm = TRUE),
    n_epochs = n(),
    .groups = "drop"
  )

##  Scale Variables

In [7]:
df_subj <- df_subj %>%
  mutate(
    delta_z = as.numeric(scale(delta)),
    theta_z = as.numeric(scale(theta)),
    alpha_z = as.numeric(scale(alpha)),
    beta_z  = as.numeric(scale(beta)),
    gamma_z = as.numeric(scale(gamma))
  )

## Permutation with permuco

In [8]:
results_no_nuisance <- list()

# ============================================================
# Models without nuisance covariates
# ============================================================

for (dv in dependent_variables) {
  
  cat("\n========================================\n")
  cat("Model without nuisance covariates\n")
  cat("Type epoch:", type_epoch, "\n")
  
  if (exists("crop") && !is.null(crop)) {
    cat("Crop:", crop, "\n")
  }
  

  cat("DV:", dv, "\n")
  cat("========================================\n")
  
  if (type_epoch == "emoc") {
    
    form_perm <- as.formula(paste(
      dv,
      "~ Condition_self * Condition_emotion +
         Error(Subject / (Condition_self * Condition_emotion))"
    ))
    
  } else if (type_epoch == "self") {
    
    form_perm <- as.formula(paste(
      dv,
      "~ Condition_self +
         Error(Subject / Condition_self)"
    ))
    
  }
  
  print(form_perm)
  
  perm_t_within <- aovperm(
    form_perm,
    data = df_subj,
    np = 5000
  )
  
  print(summary(perm_t_within))
  
  results_no_nuisance[[dv]] <- perm_t_within
}


Model without nuisance covariates
Type epoch: emoc 
DV: ACW_50 
ACW_50 ~ Condition_self * Condition_emotion + Error(Subject/(Condition_self * 
    Condition_emotion))

Resampling test using Rd_kheradPajouh_renaud to handle nuisance variables and 5000 permutations.
                                       SSn dfn       SSd dfd      MSEn
Condition_self                   2.862e-07   2 7.339e-05  58 1.431e-07
Condition_emotion                2.033e-06   2 8.477e-05  58 1.017e-06
Condition_self:Condition_emotion 4.046e-06   4 2.080e-04 116 1.012e-06
                                      MSEd      F parametric P(>F)
Condition_self                   1.265e-06 0.1131           0.8933
Condition_emotion                1.462e-06 0.6955           0.5029
Condition_self:Condition_emotion 1.793e-06 0.5641           0.6892
                                 resampled P(>F)
Condition_self                            0.9036
Condition_emotion                         0.5046
Condition_self:Condition_emotion   

In [9]:

results_with_nuisance <- list()

# ============================================================
# Models with nuisance covariates
# ============================================================

for (dv in dependent_variables) {
  
  cat("\n========================================\n")
  cat("Model with nuisance covariates\n")
  cat("Type epoch:", type_epoch, "\n")
  
  if (exists("crop") && !is.null(crop)) {
    cat("Crop:", crop, "\n")
  }
  


  cat("DV:", dv, "\n")
  cat("========================================\n")
  
  if (type_epoch == "emoc") {
    
    form_perm <- as.formula(paste(
      dv,
      "~ Condition_self * Condition_emotion +
         delta_z + theta_z + alpha_z + beta_z + gamma_z +
         Error(Subject / (Condition_self * Condition_emotion))"
    ))
    
  } else if (type_epoch == "self") {
    
    form_perm <- as.formula(paste(
      dv,
      "~ Condition_self +
         delta_z + theta_z + alpha_z + beta_z + gamma_z +
         Error(Subject / Condition_self)"
    ))
    
  }
  
  print(form_perm)
  
  mod_within_nuisance <- aovperm(
    form_perm,
    data = df_subj,
    np = 5000
  )
  
  print(summary(mod_within_nuisance))
  
  results_with_nuisance[[dv]] <- mod_within_nuisance
}


Model with nuisance covariates
Type epoch: emoc 
DV: ACW_50 
ACW_50 ~ Condition_self * Condition_emotion + delta_z + theta_z + 
    alpha_z + beta_z + gamma_z + Error(Subject/(Condition_self * 
    Condition_emotion))

Resampling test using Rd_kheradPajouh_renaud to handle nuisance variables and 5000 permutations.
                                       SSn dfn       SSd dfd      MSEn
delta_z                          1.630e-03   1 0.0067095  29 1.630e-03
theta_z                          1.486e-03   1 0.0067095  29 1.486e-03
alpha_z                          1.643e-06   1 0.0067095  29 1.643e-06
beta_z                           5.693e-06   1 0.0067095  29 5.693e-06
gamma_z                          8.902e-04   1 0.0067095  29 8.902e-04
Condition_self                   6.395e-06   2 0.0001864  58 3.197e-06
Condition_emotion                1.463e-05   2 0.0004945  58 7.313e-06
Condition_self:Condition_emotion 7.066e-06   4 0.0007821 116 1.767e-06
                                      MSEd  

# Post hocs

In [10]:
run_posthoc_from_significant_conditions <- function(model,
                                                    data,
                                                    dv,
                                                    covariates = NULL,
                                                    subject = "Subject",
                                                    alpha = 0.05,
                                                    np = 5000,
                                                    p_adjust_method = "holm",
                                                    self_factor = "Condition_self",
                                                    emotion_factor = "Condition_emotion") {
  
  # ============================================================
  # Helper: extract lmperm coefficient for the tested factor
  # ============================================================
  
  extract_lmperm_result <- function(mod_lmperm, factor_name) {
    
    # Extract lmperm summary table
    tab_lm <- as.data.frame(summary(mod_lmperm))
    
    # Add coefficient names as a column
    tab_lm$term <- rownames(tab_lm)
    
    # Use lmperm columns explicitly
    estimate_col <- "Estimate"
    t_col <- "t value"
    p_left_col <- "resampled Pr(<t)"
    p_right_col <- "resampled Pr(>t)"
    p_two_col <- "resampled Pr(>|t|)"
    
    # Find the coefficient corresponding to the tested factor
    coef_row <- grep(paste0("^", factor_name), tab_lm$term)
    
    if (length(coef_row) != 1) {
      stop("Could not uniquely identify the coefficient for: ", factor_name)
    }
    
    out <- list(
      estimate = tab_lm[[estimate_col]][coef_row],
      t_value = tab_lm[[t_col]][coef_row],
      p_left = tab_lm[[p_left_col]][coef_row],
      p_right = tab_lm[[p_right_col]][coef_row],
      p_two_tailed = tab_lm[[p_two_col]][coef_row],
      term = tab_lm$term[coef_row]
    )
    
    return(out)
  }
  
  
  # ============================================================
  # Extract significant effects from the global aovperm model
  # ============================================================
  
  tab <- summary(model)
  
  # Find significant effects based on the resampled p-value
  sig_effects <- rownames(tab)[tab[["resampled P(>F)"]] < alpha]
  
  # Detect significant Condition effects
  sig_conditions <- sig_effects[grepl("condition", sig_effects, ignore.case = TRUE)]
  
  # Detect whether the interaction is significant
  interaction_name <- paste0(self_factor, ":", emotion_factor)
  interaction_name_alt <- paste0(emotion_factor, ":", self_factor)
  
  interaction_is_significant <- any(sig_conditions %in% c(interaction_name, interaction_name_alt))
  
  posthoc_results <- list()
  
  
  # ============================================================
  # CASE 1: Significant interaction
  # Run simple effects: emotion within each self condition
  # ============================================================
  
  if (interaction_is_significant) {
    
    cat("\n========================================\n")
    cat("Significant interaction detected.\n")
    cat("Running lmperm simple effects:", emotion_factor, "within each", self_factor, "\n")
    cat("========================================\n")
    
    for (self_level in levels(data[[self_factor]])) {
      
      # Subset data to one self condition
      df_self <- subset(data, data[[self_factor]] == self_level)
      df_self[[emotion_factor]] <- droplevels(df_self[[emotion_factor]])
      
      # Skip if emotion has fewer than 3 levels
      if (length(levels(df_self[[emotion_factor]])) < 3) {
        message("Skipping ", self_level, ": fewer than 3 emotion levels.")
        next
      }
      
      # Generate all pairwise emotion comparisons within this self condition
      emotion_pairs <- combn(levels(df_self[[emotion_factor]]), 2, simplify = FALSE)
      
      estimates <- c()
      t_values <- c()
      p_left <- c()
      p_right <- c()
      p_two_tailed <- c()
      tested_terms <- c()
      level_1_values <- c()
      level_2_values <- c()
      models <- list()
      
      for (pair in emotion_pairs) {
        
        comparison_name <- paste(self_level, paste(pair, collapse = "_vs_"), sep = "__")
        
        # Keep only the two emotion levels being compared
        df_pair <- subset(df_self, df_self[[emotion_factor]] %in% pair)
        df_pair[[emotion_factor]] <- droplevels(df_pair[[emotion_factor]])
        
        # Set first level as reference
        # With treatment contrasts, estimate = pair[2] - pair[1]
        df_pair[[emotion_factor]] <- relevel(df_pair[[emotion_factor]], ref = pair[1])
        contrasts(df_pair[[emotion_factor]]) <- contr.treatment(2)
        
        # Build lmperm formula
        rhs_terms <- c(
          emotion_factor,
          covariates,
          paste0("factor(", subject, ")")
        )
        
        rhs_terms <- rhs_terms[!is.na(rhs_terms) & rhs_terms != ""]
        
        form_posthoc <- as.formula(
          paste0(
            dv, " ~ ",
            paste(rhs_terms, collapse = " + ")
          )
        )
        
        # Run lmperm for this simple-effect pairwise comparison
        mod_pair <- lmperm(
          form_posthoc,
          data = df_pair,
          np = np
        )
        
        # Extract estimate, t-value and permutation p-values
        res_pair <- extract_lmperm_result(
          mod_lmperm = mod_pair,
          factor_name = emotion_factor
        )
        
        estimates[comparison_name] <- res_pair$estimate
        t_values[comparison_name] <- res_pair$t_value
        p_left[comparison_name] <- res_pair$p_left
        p_right[comparison_name] <- res_pair$p_right
        p_two_tailed[comparison_name] <- res_pair$p_two_tailed
        tested_terms[comparison_name] <- res_pair$term
        level_1_values[comparison_name] <- pair[1]
        level_2_values[comparison_name] <- pair[2]
        
        # Store the model
        models[[comparison_name]] <- mod_pair
      }
      
      # Adjust p-values within this self condition
      p_left_adj <- p.adjust(p_left, method = p_adjust_method)
      p_right_adj <- p.adjust(p_right, method = p_adjust_method)
      p_two_tailed_adj <- p.adjust(p_two_tailed, method = p_adjust_method)
      
      # Store results in a clean table
      results_table <- data.frame(
        simple_effect = paste(emotion_factor, "within", self_factor),
        self_level = self_level,
        comparison = names(p_two_tailed),
        level_1 = as.character(level_1_values),
        level_2 = as.character(level_2_values),
        tested_term = as.character(tested_terms),
        estimate_level2_minus_level1 = as.numeric(estimates),
        estimate_level1_minus_level2 = -as.numeric(estimates),
        t_value_level2_minus_level1 = as.numeric(t_values),
        t_value_level1_minus_level2 = -as.numeric(t_values),
        p_left = as.numeric(p_left),
        p_right = as.numeric(p_right),
        p_two_tailed = as.numeric(p_two_tailed),
        p_left_holm = as.numeric(p_left_adj),
        p_right_holm = as.numeric(p_right_adj),
        p_two_tailed_holm = as.numeric(p_two_tailed_adj),
        significant_two_tailed_holm = as.numeric(p_two_tailed_adj) < alpha,
        row.names = NULL
      )
      
      # Create an interpretation table
      interpretation_table <- data.frame(
        Self_condition = self_level,
        Comparison = names(p_two_tailed),
        Level_1 = as.character(level_1_values),
        Level_2 = as.character(level_2_values),
        Tested_term = as.character(tested_terms),
        Estimate_level2_minus_level1 = round(as.numeric(estimates), 6),
        Estimate_level1_minus_level2 = round(-as.numeric(estimates), 6),
        t_value_level2_minus_level1 = round(as.numeric(t_values), 4),
        t_value_level1_minus_level2 = round(-as.numeric(t_values), 4),
        p_left = round(as.numeric(p_left), 4),
        p_right = round(as.numeric(p_right), 4),
        p_two_tailed = round(as.numeric(p_two_tailed), 4),
        p_left_holm = round(as.numeric(p_left_adj), 4),
        p_right_holm = round(as.numeric(p_right_adj), 4),
        p_two_tailed_holm = round(as.numeric(p_two_tailed_adj), 4),
        Result = ifelse(
          as.numeric(p_two_tailed_adj) < alpha,
          "Significant",
          "Not significant"
        ),
        check.names = FALSE,
        row.names = NULL
      )
      
      # Print interpretation table
      cat("\n========================================\n")
      cat("Post-hoc interaction interpretation using lmperm\n")
      cat("DV:", dv, "\n")
      cat("Simple effect:", emotion_factor, "within", self_factor, "=", self_level, "\n")
      cat("Estimate direction: level_2 - level_1\n")
      cat("P-value adjustment:", p_adjust_method, "\n")
      cat("Main decision column: p_two_tailed_holm\n")
      cat("========================================\n")
      print(interpretation_table)
      
      posthoc_results[[paste0("interaction_", self_level)]] <- list(
        table = results_table,
        interpretation = interpretation_table,
        models = models
      )
    }
    
    return(posthoc_results)
  }
  
  
  # ============================================================
  # CASE 2: No significant interaction
  # Run lmperm post-hocs for significant main Condition effects
  # ============================================================
  
  sig_main_conditions <- sig_conditions[!grepl(":", sig_conditions)]
  
  if (length(sig_main_conditions) == 0) {
    message("No significant main Condition effects found.")
    return(NULL)
  }
  
  for (factor_name in sig_main_conditions) {
    
    # Skip factors with fewer than 3 levels, because pairwise post-hocs are not needed
    if (length(levels(data[[factor_name]])) < 3) {
      message("Skipping ", factor_name, ": fewer than 3 levels.")
      next
    }
    
    # Generate all pairwise comparisons
    factor_pairs <- combn(levels(data[[factor_name]]), 2, simplify = FALSE)
    
    estimates <- c()
    t_values <- c()
    p_left <- c()
    p_right <- c()
    p_two_tailed <- c()
    tested_terms <- c()
    level_1_values <- c()
    level_2_values <- c()
    models <- list()
    
    for (pair in factor_pairs) {
      
      comparison_name <- paste(pair, collapse = "_vs_")
      
      # Keep only the two levels being compared
      df_pair <- subset(data, data[[factor_name]] %in% pair)
      df_pair[[factor_name]] <- droplevels(df_pair[[factor_name]])
      
      # Set first level as reference
      # With treatment contrasts, estimate = pair[2] - pair[1]
      df_pair[[factor_name]] <- relevel(df_pair[[factor_name]], ref = pair[1])
      contrasts(df_pair[[factor_name]]) <- contr.treatment(2)
      
      # Build lmperm formula
      rhs_terms <- c(
        factor_name,
        covariates,
        paste0("factor(", subject, ")")
      )
      
      rhs_terms <- rhs_terms[!is.na(rhs_terms) & rhs_terms != ""]
      
      form_posthoc <- as.formula(
        paste0(
          dv, " ~ ",
          paste(rhs_terms, collapse = " + ")
        )
      )
      
      # Run lmperm for this pairwise comparison
      mod_pair <- lmperm(
        form_posthoc,
        data = df_pair,
        np = np
      )
      
      # Extract estimate, t-value and permutation p-values
      res_pair <- extract_lmperm_result(
        mod_lmperm = mod_pair,
        factor_name = factor_name
      )
      
      estimates[comparison_name] <- res_pair$estimate
      t_values[comparison_name] <- res_pair$t_value
      p_left[comparison_name] <- res_pair$p_left
      p_right[comparison_name] <- res_pair$p_right
      p_two_tailed[comparison_name] <- res_pair$p_two_tailed
      tested_terms[comparison_name] <- res_pair$term
      level_1_values[comparison_name] <- pair[1]
      level_2_values[comparison_name] <- pair[2]
      
      # Store the model
      models[[comparison_name]] <- mod_pair
    }
    
    # Adjust p-values for multiple comparisons
    p_left_adj <- p.adjust(p_left, method = p_adjust_method)
    p_right_adj <- p.adjust(p_right, method = p_adjust_method)
    p_two_tailed_adj <- p.adjust(p_two_tailed, method = p_adjust_method)
    
    # Store results in a clean table
    results_table <- data.frame(
      factor = factor_name,
      comparison = names(p_two_tailed),
      level_1 = as.character(level_1_values),
      level_2 = as.character(level_2_values),
      tested_term = as.character(tested_terms),
      estimate_level2_minus_level1 = as.numeric(estimates),
      estimate_level1_minus_level2 = -as.numeric(estimates),
      t_value_level2_minus_level1 = as.numeric(t_values),
      t_value_level1_minus_level2 = -as.numeric(t_values),
      p_left = as.numeric(p_left),
      p_right = as.numeric(p_right),
      p_two_tailed = as.numeric(p_two_tailed),
      p_left_holm = as.numeric(p_left_adj),
      p_right_holm = as.numeric(p_right_adj),
      p_two_tailed_holm = as.numeric(p_two_tailed_adj),
      significant_two_tailed_holm = as.numeric(p_two_tailed_adj) < alpha,
      row.names = NULL
    )
    
    # Create an interpretation table
    interpretation_table <- data.frame(
      Comparison = names(p_two_tailed),
      Level_1 = as.character(level_1_values),
      Level_2 = as.character(level_2_values),
      Tested_term = as.character(tested_terms),
      Estimate_level2_minus_level1 = round(as.numeric(estimates), 6),
      Estimate_level1_minus_level2 = round(-as.numeric(estimates), 6),
      t_value_level2_minus_level1 = round(as.numeric(t_values), 4),
      t_value_level1_minus_level2 = round(-as.numeric(t_values), 4),
      p_left = round(as.numeric(p_left), 4),
      p_right = round(as.numeric(p_right), 4),
      p_two_tailed = round(as.numeric(p_two_tailed), 4),
      p_left_holm = round(as.numeric(p_left_adj), 4),
      p_right_holm = round(as.numeric(p_right_adj), 4),
      p_two_tailed_holm = round(as.numeric(p_two_tailed_adj), 4),
      Result = ifelse(
        as.numeric(p_two_tailed_adj) < alpha,
        "Significant",
        "Not significant"
      ),
      check.names = FALSE,
      row.names = NULL
    )
    
    # Print interpretation table
    cat("\n========================================\n")
    cat("Post-hoc interpretation using lmperm for:", factor_name, "\n")
    cat("DV:", dv, "\n")
    cat("Estimate direction: level_2 - level_1\n")
    cat("P-value adjustment:", p_adjust_method, "\n")
    cat("Main decision column: p_two_tailed_holm\n")
    cat("========================================\n")
    print(interpretation_table)
    
    posthoc_results[[factor_name]] <- list(
      table = results_table,
      interpretation = interpretation_table,
      models = models
    )
  }
  
  return(posthoc_results)
}

In [ ]:
# run_posthoc_from_significant_conditions <- function(model,
#                                                     data,
#                                                     dv,
#                                                     covariates = NULL,
#                                                     subject = "Subject",
#                                                     alpha = 0.05,
#                                                     np = 5000,
#                                                     p_adjust_method = "holm",
#                                                     self_factor = "Condition_self",
#                                                     emotion_factor = "Condition_emotion") {
  
#   # Extract the permutation ANOVA table
#   tab <- summary(model)
  
#   # Find significant effects based on the resampled p-value
#   sig_effects <- rownames(tab)[tab[["resampled P(>F)"]] < alpha]
  
#   # Detect significant Condition effects
#   sig_conditions <- sig_effects[grepl("condition", sig_effects, ignore.case = TRUE)]
  
#   # Detect whether the interaction is significant
#   interaction_name <- paste0(self_factor, ":", emotion_factor)
#   interaction_name_alt <- paste0(emotion_factor, ":", self_factor)
  
#   interaction_is_significant <- any(sig_conditions %in% c(interaction_name, interaction_name_alt))
  
#   posthoc_results <- list()
  
#   # ============================================================
#   # CASE 1: Significant interaction
#   # Run simple effects: emotion within each self condition
#   # ============================================================
  
#   if (interaction_is_significant) {
    
#     cat("\n========================================\n")
#     cat("Significant interaction detected.\n")
#     cat("Running simple effects:", emotion_factor, "within each", self_factor, "\n")
#     cat("========================================\n")
    
#     for (self_level in levels(data[[self_factor]])) {
      
#       # Subset data to one self condition
#       df_self <- subset(data, data[[self_factor]] == self_level)
#       df_self[[emotion_factor]] <- droplevels(df_self[[emotion_factor]])
      
#       # Skip if emotion has fewer than 3 levels
#       if (length(levels(df_self[[emotion_factor]])) < 3) {
#         message("Skipping ", self_level, ": fewer than 3 emotion levels.")
#         next
#       }
      
#       # Generate all pairwise emotion comparisons within this self condition
#       emotion_pairs <- combn(levels(df_self[[emotion_factor]]), 2, simplify = FALSE)
      
#       pvals <- c()
#       models <- list()
      
#       for (pair in emotion_pairs) {
        
#         comparison_name <- paste(self_level, paste(pair, collapse = "_vs_"), sep = "__")
        
#         # Keep only the two emotion levels being compared
#         df_pair <- subset(df_self, df_self[[emotion_factor]] %in% pair)
#         df_pair[[emotion_factor]] <- droplevels(df_pair[[emotion_factor]])
        
#         # Build the model formula
#         rhs_terms <- emotion_factor
        
#         if (!is.null(covariates)) {
#           rhs_terms <- paste(c(rhs_terms, covariates), collapse = " + ")
#         }
        
#         form_posthoc <- as.formula(
#           paste0(
#             dv, " ~ ", rhs_terms,
#             " + Error(", subject, " / ", emotion_factor, ")"
#           )
#         )
        
#         # Run the permutation model for this simple-effect pairwise comparison
#         mod_pair <- aovperm(
#           form_posthoc,
#           data = df_pair,
#           np = np
#         )
        
#         # Extract the resampled p-value for the emotion factor
#         tab_pair <- summary(mod_pair)
#         pvals[comparison_name] <- tab_pair[emotion_factor, "resampled P(>F)"]
        
#         # Store the model
#         models[[comparison_name]] <- mod_pair
#       }
      
#       # Adjust p-values within this self condition
#       p_adj <- p.adjust(pvals, method = p_adjust_method)
      
#       # Store results in a clean table
#       results_table <- data.frame(
#         simple_effect = paste(emotion_factor, "within", self_factor),
#         self_level = self_level,
#         comparison = names(pvals),
#         p_perm = as.numeric(pvals),
#         p_adjusted = as.numeric(p_adj),
#         significant = as.numeric(p_adj) < alpha,
#         row.names = NULL
#       )
      
#       # Create an interpretation table
#       interpretation_table <- data.frame(
#         Self_condition = self_level,
#         Comparison = names(pvals),
#         `p_perm` = round(as.numeric(pvals), 4),
#         `p_adjusted` = round(as.numeric(p_adj), 4),
#         Result = ifelse(
#           as.numeric(p_adj) < alpha,
#           "Significant",
#           "Not significant"
#         ),
#         check.names = FALSE,
#         row.names = NULL
#       )
      
#       # Print interpretation table
#       cat("\n========================================\n")
#       cat("Post-hoc interaction interpretation\n")
#       cat("DV:", dv, "\n")
#       cat("Simple effect:", emotion_factor, "within", self_factor, "=", self_level, "\n")
#       cat("P-value adjustment:", p_adjust_method, "\n")
#       cat("========================================\n")
#       print(interpretation_table)
      
#       posthoc_results[[paste0("interaction_", self_level)]] <- list(
#         table = results_table,
#         interpretation = interpretation_table,
#         models = models
#       )
#     }
    
#     return(posthoc_results)
#   }
  
#   # ============================================================
#   # CASE 2: No significant interaction
#   # Run post-hocs for significant main Condition effects
#   # ============================================================
  
#   sig_main_conditions <- sig_conditions[!grepl(":", sig_conditions)]
  
#   if (length(sig_main_conditions) == 0) {
#     message("No significant main Condition effects found.")
#     return(NULL)
#   }
  
#   for (factor_name in sig_main_conditions) {
    
#     # Skip factors with fewer than 3 levels, because pairwise post-hocs are not needed
#     if (length(levels(data[[factor_name]])) < 3) {
#       message("Skipping ", factor_name, ": fewer than 3 levels.")
#       next
#     }
    
#     # Generate all pairwise comparisons
#     factor_pairs <- combn(levels(data[[factor_name]]), 2, simplify = FALSE)
    
#     pvals <- c()
#     models <- list()
    
#     for (pair in factor_pairs) {
      
#       comparison_name <- paste(pair, collapse = "_vs_")
      
#       # Keep only the two levels being compared
#       df_pair <- subset(data, data[[factor_name]] %in% pair)
#       df_pair[[factor_name]] <- droplevels(df_pair[[factor_name]])
      
#       # Build the model formula
#       rhs_terms <- factor_name
      
#       if (!is.null(covariates)) {
#         rhs_terms <- paste(c(rhs_terms, covariates), collapse = " + ")
#       }
      
#       form_posthoc <- as.formula(
#         paste0(
#           dv, " ~ ", rhs_terms,
#           " + Error(", subject, " / ", factor_name, ")"
#         )
#       )
      
#       # Run the permutation model for this pairwise comparison
#       mod_pair <- aovperm(
#         form_posthoc,
#         data = df_pair,
#         np = np
#       )
      
#       # Extract the resampled p-value for the factor
#       tab_pair <- summary(mod_pair)
#       pvals[comparison_name] <- tab_pair[factor_name, "resampled P(>F)"]
      
#       # Store the model
#       models[[comparison_name]] <- mod_pair
#     }
    
#     # Adjust p-values for multiple comparisons
#     p_adj <- p.adjust(pvals, method = p_adjust_method)
    
#     # Store results in a clean table
#     results_table <- data.frame(
#       factor = factor_name,
#       comparison = names(pvals),
#       p_perm = as.numeric(pvals),
#       p_adjusted = as.numeric(p_adj),
#       significant = as.numeric(p_adj) < alpha,
#       row.names = NULL
#     )
    
#     # Create an interpretation table
#     interpretation_table <- data.frame(
#       Comparison = names(pvals),
#       `p_perm` = round(as.numeric(pvals), 4),
#       `p_adjusted` = round(as.numeric(p_adj), 4),
#       Result = ifelse(
#         as.numeric(p_adj) < alpha,
#         "Significant",
#         "Not significant"
#       ),
#       check.names = FALSE,
#       row.names = NULL
#     )
    
#     # Print interpretation table
#     cat("\n========================================\n")
#     cat("Post-hoc interpretation for:", factor_name, "\n")
#     cat("DV:", dv, "\n")
#     cat("P-value adjustment:", p_adjust_method, "\n")
#     cat("========================================\n")
#     print(interpretation_table)
    
#     posthoc_results[[factor_name]] <- list(
#       table = results_table,
#       interpretation = interpretation_table,
#       models = models
#     )
#   }
  
#   return(posthoc_results)
# }

In [11]:
# ============================================================
# Post-hoc parameters
# ============================================================

covariates <- c("delta_z", "theta_z", "alpha_z", "beta_z", "gamma_z")
subject <- "Subject"
alpha <- 0.05
np <- 5000
p_adjust_method <- "holm"

posthoc_results_all <- list()


# ============================================================
# Run post-hocs for each dependent variable
# ============================================================

for (dv in dependent_variables) {
  
  cat("\n========================================\n")
  cat("Running post-hocs\n")
  cat("DV:", dv, "\n")
  cat("========================================\n")
  
  # Get the model already stored for this DV
  model <- results_with_nuisance[[dv]]
  
  # Run automatic post-hocs
  posthoc_results_all[[dv]] <- run_posthoc_from_significant_conditions(
    model = model,
    data = df_subj,
    dv = dv,
    covariates = covariates,
    subject = subject,
    alpha = alpha,
    np = np,
    p_adjust_method = p_adjust_method
  )
}

posthoc_results_all


Running post-hocs
DV: ACW_50 


No significant main Condition effects found.




Running post-hocs
DV: ACW_0 

Post-hoc interpretation using lmperm for: Condition_emotion 
DV: ACW_0 
Estimate direction: level_2 - level_1
P-value adjustment: holm 
Main decision column: p_two_tailed_holm
            Comparison  Level_1  Level_2        Tested_term
1  negative_vs_neutral negative  neutral Condition_emotion2
2 negative_vs_positive negative positive Condition_emotion2
3  neutral_vs_positive  neutral positive Condition_emotion2
  Estimate_level2_minus_level1 Estimate_level1_minus_level2
1                    -0.005407                     0.005407
2                     0.000503                    -0.000503
3                     0.004040                    -0.004040
  t_value_level2_minus_level1 t_value_level1_minus_level2 p_left p_right
1                     -3.8702                      3.8702 0.0004  0.9998
2                      0.3783                     -0.3783 0.6518  0.3484
3                      3.0191                     -3.0191 0.9992  0.0010
  p_two_tailed p_left

$ACW_0
$ACW_0$Condition_emotion
$ACW_0$Condition_emotion$table
             factor           comparison  level_1  level_2        tested_term
1 Condition_emotion  negative_vs_neutral negative  neutral Condition_emotion2
2 Condition_emotion negative_vs_positive negative positive Condition_emotion2
3 Condition_emotion  neutral_vs_positive  neutral positive Condition_emotion2
  estimate_level2_minus_level1 estimate_level1_minus_level2
1                -0.0054069613                 0.0054069613
2                 0.0005028029                -0.0005028029
3                 0.0040396157                -0.0040396157
  t_value_level2_minus_level1 t_value_level1_minus_level2 p_left p_right
1                  -3.8702092                   3.8702092 0.0004  0.9998
2                   0.3782768                  -0.3782768 0.6518  0.3484
3                   3.0190670                  -3.0190670 0.9992  0.0010
  p_two_tailed p_left_holm p_right_holm p_two_tailed_holm
1       0.0004      0.0012       0.

In [12]:
for (dv in dependent_variables) {
  
  cat("\n========================================\n")
  cat("Post-hoc interpretation\n")
  cat("DV:", dv, "\n")
  cat("========================================\n")
  
  if (is.null(posthoc_results_all[[dv]])) {
    cat("No post-hoc results for this DV.\n")
    next
  }
  
  for (effect_name in names(posthoc_results_all[[dv]])) {
    
    cat("\nEffect:", effect_name, "\n")
    
    print(posthoc_results_all[[dv]][[effect_name]]$interpretation)
  }
}


Post-hoc interpretation
DV: ACW_50 
No post-hoc results for this DV.

Post-hoc interpretation
DV: ACW_0 

Effect: Condition_emotion 
            Comparison  Level_1  Level_2        Tested_term
1  negative_vs_neutral negative  neutral Condition_emotion2
2 negative_vs_positive negative positive Condition_emotion2
3  neutral_vs_positive  neutral positive Condition_emotion2
  Estimate_level2_minus_level1 Estimate_level1_minus_level2
1                    -0.005407                     0.005407
2                     0.000503                    -0.000503
3                     0.004040                    -0.004040
  t_value_level2_minus_level1 t_value_level1_minus_level2 p_left p_right
1                     -3.8702                      3.8702 0.0004  0.9998
2                      0.3783                     -0.3783 0.6518  0.3484
3                      3.0191                     -3.0191 0.9992  0.0010
  p_two_tailed p_left_holm p_right_holm p_two_tailed_holm          Result
1       0.0004      

In [ ]:
# ============================================================
# lmperm post-hoc: self - unknown for both DVs
# Unknown is the reference/intercept
# ============================================================

posthoc_self_unknown <- list()

for (dv in dependent_variables) {
  
  cat("\n========================================\n")
  cat("lmperm post-hoc: self - unknown\n")
  cat("DV:", dv, "\n")
  cat("========================================\n")
  
  # Keep only self and unknown
  df_pair <- subset(df_subj, Condition_self %in% c("self", "unknown"))
  df_pair$Condition_self <- droplevels(df_pair$Condition_self)
  
  # Set unknown as the reference/intercept
  df_pair$Condition_self <- relevel(df_pair$Condition_self, ref = "unknown")
  
  # Build formula
  form_lmperm <- as.formula(
    paste0(
      dv,
      " ~ Condition_self + ",
      "delta_z + theta_z + alpha_z + beta_z + gamma_z + ",
      "factor(Subject)"
    )
  )
  
  print(form_lmperm)
  
  # Run lmperm
  mod_pair <- lmperm(
    form_lmperm,
    data = df_pair,
    np = 5000
  )
  
  print(summary(mod_pair))
  
  # Store model
  posthoc_self_unknown[[dv]] <- mod_pair
}

# Model Evaluation

In [ ]:
# ============================================================
# Model evaluation / diagnostics for permuco model
# ============================================================

cat("\n----------------------------------------\n")
cat("Evaluating model for DV:", dv, "\n")
cat("----------------------------------------\n")

# 1. Basic summary: ANOVA table + parametric and permutation p-values
cat("\n[1] summary(aovperm model)\n")
model_summary <- summary(mod_within_nuisance)
print(model_summary)

# 2. Inspect available components in the permuco object
cat("\n[2] Available components in the model object\n")
print(names(mod_within_nuisance))

# 3. Inspect object class
cat("\n[3] Model class\n")
print(class(mod_within_nuisance))

# 4. Plot permutation distributions
#    permuco's plot.lmperm shows the density of permuted statistics
#    and the observed test statistic.
cat("\n[4] Permutation distribution plots\n")

tryCatch({
  plot(mod_within_nuisance)
}, error = function(e) {
  cat("Could not plot full model for DV:", dv, "\n")
  cat("Error:", e$message, "\n")
})

# 5. Optional: plot each effect separately if effect names are available
#    Different permuco versions may store the ANOVA table differently,
#    so this is written defensively.

cat("\n[5] Trying effect-specific plots\n")

effect_names <- NULL

# Try to recover effect names from summary object
if (is.list(model_summary)) {
  possible_tables <- model_summary[sapply(model_summary, is.data.frame)]
  
  if (length(possible_tables) > 0) {
    effect_names <- unique(unlist(lapply(possible_tables, rownames)))
  }
}

# Remove residual/error rows if present
effect_names <- effect_names[
  !is.na(effect_names) &
    !grepl("Residual|Error|Within|Subject", effect_names, ignore.case = TRUE)
]

print(effect_names)

if (!is.null(effect_names) && length(effect_names) > 0) {
  for (eff in effect_names) {
    cat("\nPlotting effect:", eff, "\n")
    
    tryCatch({
      plot(mod_within_nuisance, effect = eff)
    }, error = function(e) {
      cat("Could not plot effect:", eff, "\n")
      cat("Error:", e$message, "\n")
    })
  }
} else {
  cat("No effect names could be extracted automatically.\n")
}

# 6. Optional: classical residual diagnostics if residuals/fitted are available
#    These are not the main permutation inference, but useful to inspect model behavior.

cat("\n[6] Residual and fitted-value diagnostics, if available\n")

res <- tryCatch(residuals(mod_within_nuisance), error = function(e) NULL)
fit <- tryCatch(fitted(mod_within_nuisance), error = function(e) NULL)

if (!is.null(res)) {
  cat("Residual summary:\n")
  print(summary(res))
  
  hist(
    res,
    main = paste("Residuals -", dv),
    xlab = "Residuals"
  )
  
  qqnorm(res, main = paste("QQ plot residuals -", dv))
  qqline(res)
} else {
  cat("Residuals not directly available for this permuco object.\n")
}

if (!is.null(res) && !is.null(fit)) {
  plot(
    fit, res,
    main = paste("Residuals vs fitted -", dv),
    xlab = "Fitted values",
    ylab = "Residuals"
  )
  abline(h = 0, lty = 2)
} else {
  cat("Fitted values not directly available for this permuco object.\n")
}

# 7. Store evaluation output
results_with_nuisance[[dv]] <- list(
  model = mod_within_nuisance,
  summary = model_summary,
  components = names(mod_within_nuisance),
  class = class(mod_within_nuisance),
  residuals = res,
  fitted = fit
)

In [ ]:
cat("\n[4] Permutation distribution plots\n")

effect_names <- rownames(mod_within_nuisance$table)

effect_names <- effect_names[
  !is.na(effect_names) &
    !grepl("Residual|Error|Within|Subject", effect_names, ignore.case = TRUE)
]

print(effect_names)

for (eff in effect_names) {
  cat("\nPlotting permutation distribution for effect:", eff, "\n")
  
  tryCatch({
    plot(mod_within_nuisance, effect = eff)
  }, error = function(e) {
    cat("Could not plot effect:", eff, "\n")
    cat("Error:", e$message, "\n")
  })
}

In [ ]:
# ============================================================
# Exchangeability assessment for permuco model
# ============================================================

cat("\n----------------------------------------\n")
cat("Exchangeability assessment for DV:", dv, "\n")
cat("----------------------------------------\n")

# 1. Method used by permuco
cat("\n[1] Permutation method used by permuco:\n")
print(mod_within_nuisance$method)

# 2. Number of permutations
cat("\n[2] Number of permutations:\n")
print(mod_within_nuisance$np)

# 3. Check the permutation matrix / transformation object
cat("\n[3] Permutation object P:\n")
print(class(mod_within_nuisance$P))
print(dim(mod_within_nuisance$P))

# 4. Check whether the model includes repeated-measures structure
cat("\n[4] Model formula:\n")
print(form_perm)

cat("\n[5] Interpretation:\n")

if (grepl("kheradPajouh_renaud", mod_within_nuisance$method)) {
  cat(
    "The model uses a Kherad-Pajouh & Renaud permutation method,\n",
    "which is appropriate for models with Error(...) / repeated-measures structure.\n",
    "Exchangeability is not assessed through normality of residuals.\n",
    "Instead, the method defines a restricted/transformed residual permutation scheme\n",
    "compatible with the repeated-measures design and nuisance covariates.\n"
  )
} else {
  cat(
    "The model does not appear to use a Kherad-Pajouh & Renaud method.\n",
    "Check whether the selected method is appropriate for nuisance covariates\n",
    "and repeated-measures structure.\n"
  )
}

# 6. Optional sensitivity analysis: compare Rd and Rde
cat("\n[6] Sensitivity analysis: Rd vs Rde\n")

mod_Rd <- aovperm(
  form_perm,
  data = df_subj,
  np = 5000,
  method = "Rd_kheradPajouh_renaud"
)

mod_Rde <- aovperm(
  form_perm,
  data = df_subj,
  np = 5000,
  method = "Rde_kheradPajouh_renaud"
)

cat("\nRd_kheradPajouh_renaud:\n")
print(summary(mod_Rd))

cat("\nRde_kheradPajouh_renaud:\n")
print(summary(mod_Rde))

# 7. Store exchangeability/sensitivity information
exchangeability_check <- list(
  method_used = mod_within_nuisance$method,
  np = mod_within_nuisance$np,
  P_class = class(mod_within_nuisance$P),
  P_dim = dim(mod_within_nuisance$P),
  Rd_summary = summary(mod_Rd),
  Rde_summary = summary(mod_Rde)
)

In [ ]:
# ============================================================
# Auxiliary residual diagnostics for exchangeability
# ============================================================

cat("\n----------------------------------------\n")
cat("Auxiliary residual exchangeability check for DV:", dv, "\n")
cat("----------------------------------------\n")

if (type_epoch == "emoc") {
  
  form_lm_diag <- as.formula(paste(
    dv,
    "~ Condition_self * Condition_emotion +
       delta_z + theta_z + alpha_z + beta_z + gamma_z +
       Subject"
  ))
  
  cell_factor <- interaction(
    df_subj$Condition_self,
    df_subj$Condition_emotion,
    drop = TRUE
  )
  
} else if (type_epoch == "self") {
  
  form_lm_diag <- as.formula(paste(
    dv,
    "~ Condition_self +
       delta_z + theta_z + alpha_z + beta_z + gamma_z +
       Subject"
  ))
  
  cell_factor <- df_subj$Condition_self
}

lm_diag <- lm(form_lm_diag, data = df_subj)

df_subj$resid_diag <- residuals(lm_diag)
df_subj$fitted_diag <- fitted(lm_diag)
df_subj$cell_diag <- cell_factor

cat("\nResidual summary:\n")
print(summary(df_subj$resid_diag))

cat("\nResidual SD by experimental cell:\n")
print(tapply(df_subj$resid_diag, df_subj$cell_diag, sd, na.rm = TRUE))

cat("\nResidual mean by experimental cell:\n")
print(tapply(df_subj$resid_diag, df_subj$cell_diag, mean, na.rm = TRUE))

cat("\nResidual SD by subject:\n")
print(tapply(df_subj$resid_diag, df_subj$Subject, sd, na.rm = TRUE))

In [ ]:
mod_within_nuisance <- aovperm(
  form_perm,
  data = df_subj,
  np = 5000,
  method = "Rde_kheradPajouh_renaud"
)

In [ ]:
summary(mod_within_nuisance)

In [ ]:
# ============================================================
# Auxiliary residual diagnostics for exchangeability plausibility
# ============================================================

cat("\n----------------------------------------\n")
cat("Auxiliary residual diagnostics for DV:", dv, "\n")
cat("----------------------------------------\n")

# ------------------------------------------------------------
# 1. Auxiliary parametric model
# ------------------------------------------------------------

if (type_epoch == "emoc") {
  
  form_lm_diag <- as.formula(paste(
    dv,
    "~ Condition_self * Condition_emotion +
       delta_z + theta_z + alpha_z + beta_z + gamma_z +
       Subject"
  ))
  
  df_subj$cell_diag <- interaction(
    df_subj$Condition_self,
    df_subj$Condition_emotion,
    drop = TRUE
  )
  
} else if (type_epoch == "self") {
  
  form_lm_diag <- as.formula(paste(
    dv,
    "~ Condition_self +
       delta_z + theta_z + alpha_z + beta_z + gamma_z +
       Subject"
  ))
  
  df_subj$cell_diag <- df_subj$Condition_self
}

lm_diag <- lm(form_lm_diag, data = df_subj)

df_subj$resid_diag <- residuals(lm_diag)
df_subj$fitted_diag <- fitted(lm_diag)

# ------------------------------------------------------------
# 2. General residual summary
# ------------------------------------------------------------

cat("\n[1] General residual summary\n")
print(summary(df_subj$resid_diag))

cat("\nResidual SD overall:\n")
print(sd(df_subj$resid_diag, na.rm = TRUE))

cat("\nResidual MAD overall:\n")
print(mad(df_subj$resid_diag, na.rm = TRUE))

# ------------------------------------------------------------
# 3. Residual means and SDs by experimental cell
# ------------------------------------------------------------

cat("\n[2] Residual mean by experimental cell\n")
resid_mean_cell <- tapply(
  df_subj$resid_diag,
  df_subj$cell_diag,
  mean,
  na.rm = TRUE
)
print(resid_mean_cell)

cat("\n[3] Residual SD by experimental cell\n")
resid_sd_cell <- tapply(
  df_subj$resid_diag,
  df_subj$cell_diag,
  sd,
  na.rm = TRUE
)
print(resid_sd_cell)

cat("\nRange of residual SDs by cell:\n")
print(range(resid_sd_cell, na.rm = TRUE))

cat("\nMax/min ratio of residual SDs by cell:\n")
print(max(resid_sd_cell, na.rm = TRUE) / min(resid_sd_cell, na.rm = TRUE))

# ------------------------------------------------------------
# 4. Residual means and SDs by subject
# ------------------------------------------------------------

cat("\n[4] Residual mean by subject\n")
resid_mean_subject <- tapply(
  df_subj$resid_diag,
  df_subj$Subject,
  mean,
  na.rm = TRUE
)
print(resid_mean_subject)

cat("\n[5] Residual SD by subject\n")
resid_sd_subject <- tapply(
  df_subj$resid_diag,
  df_subj$Subject,
  sd,
  na.rm = TRUE
)
print(resid_sd_subject)

cat("\nRange of residual SDs by subject:\n")
print(range(resid_sd_subject, na.rm = TRUE))

cat("\nMax/min ratio of residual SDs by subject:\n")
print(max(resid_sd_subject, na.rm = TRUE) / min(resid_sd_subject, na.rm = TRUE))

# ------------------------------------------------------------
# 5. Robust dispersion tests
# ------------------------------------------------------------

cat("\n[6] Fligner-Killeen test: residual dispersion by experimental cell\n")
print(
  fligner.test(resid_diag ~ cell_diag, data = df_subj)
)

cat("\n[7] Fligner-Killeen test: residual dispersion by subject\n")
print(
  fligner.test(resid_diag ~ Subject, data = df_subj)
)

# ------------------------------------------------------------
# 6. Outlier check
# ------------------------------------------------------------

cat("\n[8] Standardized residuals\n")

df_subj$std_resid_diag <- rstandard(lm_diag)

print(summary(df_subj$std_resid_diag))

cat("\nNumber of |standardized residuals| > 2:\n")
print(sum(abs(df_subj$std_resid_diag) > 2, na.rm = TRUE))

cat("\nNumber of |standardized residuals| > 3:\n")
print(sum(abs(df_subj$std_resid_diag) > 3, na.rm = TRUE))

cat("\nRows with |standardized residuals| > 3:\n")
print(
  df_subj[abs(df_subj$std_resid_diag) > 3, 
          c("Subject", "Condition_self", "Condition_emotion",
            "resid_diag", "std_resid_diag")]
)

# ------------------------------------------------------------
# 7. Visual diagnostics
# ------------------------------------------------------------

par(mfrow = c(2, 2))

boxplot(
  resid_diag ~ cell_diag,
  data = df_subj,
  main = paste("Residuals by condition cell -", dv),
  xlab = "Condition cell",
  ylab = "Auxiliary residuals",
  las = 2
)
abline(h = 0, lty = 2)

boxplot(
  resid_diag ~ Subject,
  data = df_subj,
  main = paste("Residuals by subject -", dv),
  xlab = "Subject",
  ylab = "Auxiliary residuals",
  las = 2
)
abline(h = 0, lty = 2)

plot(
  df_subj$fitted_diag,
  df_subj$resid_diag,
  main = paste("Residuals vs fitted -", dv),
  xlab = "Fitted values",
  ylab = "Auxiliary residuals"
)
abline(h = 0, lty = 2)

qqnorm(
  df_subj$resid_diag,
  main = paste("QQ plot residuals -", dv)
)
qqline(df_subj$resid_diag)

par(mfrow = c(1, 1))

# ------------------------------------------------------------
# 8. Store diagnostics
# ------------------------------------------------------------

aux_resid_diagnostics <- list(
  lm_diag = lm_diag,
  residual_summary = summary(df_subj$resid_diag),
  resid_mean_cell = resid_mean_cell,
  resid_sd_cell = resid_sd_cell,
  resid_sd_cell_ratio = max(resid_sd_cell, na.rm = TRUE) / 
    min(resid_sd_cell, na.rm = TRUE),
  resid_mean_subject = resid_mean_subject,
  resid_sd_subject = resid_sd_subject,
  resid_sd_subject_ratio = max(resid_sd_subject, na.rm = TRUE) / 
    min(resid_sd_subject, na.rm = TRUE),
  fligner_cell = fligner.test(resid_diag ~ cell_diag, data = df_subj),
  fligner_subject = fligner.test(resid_diag ~ Subject, data = df_subj),
  n_std_resid_gt_2 = sum(abs(df_subj$std_resid_diag) > 2, na.rm = TRUE),
  n_std_resid_gt_3 = sum(abs(df_subj$std_resid_diag) > 3, na.rm = TRUE)
)

In [ ]:
tab <- with(df_subj, table(Subject, Condition_self, Condition_emotion))
range(tab)

# testing permuco with lm

In [ ]:
diagnostics_with_nuisance <- list()

# ============================================================
# Diagnostics for permuco models with nuisance covariates
# ============================================================

for (dv in dependent_variables) {
  
  cat("\n========================================\n")
  cat("Diagnostics for model with nuisance covariates\n")
  cat("Type epoch:", type_epoch, "\n")
  
  if (exists("crop") && !is.null(crop)) {
    cat("Crop:", crop, "\n")
  }
  
  cat("DV:", dv, "\n")
  cat("========================================\n")
  
  mod_perm <- results_with_nuisance[[dv]]
  
  # ------------------------------------------------------------
  # Try direct diagnostics from the aovperm/lmperm object
  # ------------------------------------------------------------
  
  cat("\n--- Object class ---\n")
  print(class(mod_perm))
  
  cat("\n--- Available object names ---\n")
  print(names(mod_perm))
  
  cat("\n--- Trying plot(mod_perm) ---\n")
  try(plot(mod_perm), silent = TRUE)
  
  res_perm <- try(residuals(mod_perm), silent = TRUE)
  fit_perm <- try(fitted(mod_perm), silent = TRUE)
  
  has_direct_residuals <- !inherits(res_perm, "try-error") &&
    !inherits(fit_perm, "try-error") &&
    length(res_perm) == nrow(df_subj) &&
    length(fit_perm) == nrow(df_subj)
  
  # ------------------------------------------------------------
  # If residuals can be extracted directly from aovperm
  # ------------------------------------------------------------
  
  if (has_direct_residuals) {
    
    cat("\nUsing residuals extracted directly from aovperm object.\n")
    
    df_diag <- df_subj %>%
      mutate(
        fitted = as.numeric(fit_perm),
        residual = as.numeric(res_perm),
        scaled_residual = as.numeric(scale(residual))
      )
    
  } else {
    
    cat("\nCould not extract clean residuals from aovperm object.\n")
    cat("Fitting equivalent lm model for diagnostic purposes only.\n")
    
    # ------------------------------------------------------------
    # Equivalent lm model for diagnostics only
    # ------------------------------------------------------------
    
    if (type_epoch == "emoc") {
      
      form_lm_diag <- as.formula(paste(
        dv,
        "~ Subject +
           Condition_self * Condition_emotion +
           delta_z + theta_z + alpha_z + beta_z + gamma_z"
      ))
      
    } else if (type_epoch == "self") {
      
      form_lm_diag <- as.formula(paste(
        dv,
        "~ Subject +
           Condition_self +
           delta_z + theta_z + alpha_z + beta_z + gamma_z"
      ))
      
    }
    
    print(form_lm_diag)
    
    mod_lm_diag <- lm(
      form_lm_diag,
      data = df_subj
    )
    
    df_diag <- df_subj %>%
      mutate(
        fitted = fitted(mod_lm_diag),
        residual = residuals(mod_lm_diag),
        scaled_residual = rstandard(mod_lm_diag),
        cooks = cooks.distance(mod_lm_diag),
        leverage = hatvalues(mod_lm_diag)
      )
    
    cat("\n--- lm diagnostic plots ---\n")
    par(mfrow = c(2, 2))
    plot(mod_lm_diag)
    par(mfrow = c(1, 1))
    
    cat("\n--- Collinearity check ---\n")
    if (requireNamespace("car", quietly = TRUE)) {
      print(car::vif(mod_lm_diag))
    } else {
      cat("Package 'car' not installed. Skipping VIF.\n")
    }
  }
  
  # ------------------------------------------------------------
  # General residual plots
  # ------------------------------------------------------------
  
  cat("\n--- Residual plots ---\n")
  
  par(mfrow = c(2, 2))
  
  plot(
    df_diag$fitted,
    df_diag$residual,
    xlab = "Fitted values",
    ylab = "Residuals",
    main = paste("Residuals vs fitted -", dv)
  )
  abline(h = 0, lty = 2)
  
  qqnorm(
    df_diag$residual,
    main = paste("QQ plot residuals -", dv)
  )
  qqline(df_diag$residual)
  
  hist(
    df_diag$residual,
    breaks = 20,
    main = paste("Histogram residuals -", dv),
    xlab = "Residuals"
  )
  
  plot(
    df_diag$fitted,
    sqrt(abs(df_diag$scaled_residual)),
    xlab = "Fitted values",
    ylab = "sqrt(|scaled residuals|)",
    main = paste("Scale-location -", dv)
  )
  
  par(mfrow = c(1, 1))
  
  # ------------------------------------------------------------
  # Residuals by experimental condition
  # ------------------------------------------------------------
  
  cat("\n--- Residuals by condition ---\n")
  
  if (type_epoch == "emoc") {
    
    boxplot(
      residual ~ Condition_self * Condition_emotion,
      data = df_diag,
      las = 2,
      main = paste("Residuals by Condition_self x Condition_emotion -", dv),
      ylab = "Residuals"
    )
    abline(h = 0, lty = 2)
    
  } else if (type_epoch == "self") {
    
    boxplot(
      residual ~ Condition_self,
      data = df_diag,
      las = 2,
      main = paste("Residuals by Condition_self -", dv),
      ylab = "Residuals"
    )
    abline(h = 0, lty = 2)
  }
  
  # ------------------------------------------------------------
  # Residuals by subject
  # ------------------------------------------------------------
  
  cat("\n--- Residuals by subject ---\n")
  
  boxplot(
    residual ~ Subject,
    data = df_diag,
    las = 2,
    main = paste("Residuals by subject -", dv),
    ylab = "Residuals"
  )
  abline(h = 0, lty = 2)
  
  # ------------------------------------------------------------
  # Largest residuals / potential outliers
  # ------------------------------------------------------------
  
  cat("\n--- Largest absolute scaled residuals ---\n")
  
  cols_to_show <- c(
    "Subject",
    "Condition_self",
    "Condition_emotion",
    dv,
    "delta_z", "theta_z", "alpha_z", "beta_z", "gamma_z",
    "fitted", "residual", "scaled_residual"
  )
  
  cols_to_show <- cols_to_show[cols_to_show %in% names(df_diag)]
  
  print(
    df_diag %>%
      arrange(desc(abs(scaled_residual))) %>%
      select(all_of(cols_to_show)) %>%
      head(15)
  )
  
  # ------------------------------------------------------------
  # Residual summaries by condition
  # ------------------------------------------------------------
  
  cat("\n--- Residual summary by condition ---\n")
  
  if (type_epoch == "emoc") {
    
    print(
      df_diag %>%
        group_by(Condition_self, Condition_emotion) %>%
        summarise(
          mean_residual = mean(residual, na.rm = TRUE),
          sd_residual = sd(residual, na.rm = TRUE),
          median_residual = median(residual, na.rm = TRUE),
          n = n(),
          .groups = "drop"
        )
    )
    
  } else if (type_epoch == "self") {
    
    print(
      df_diag %>%
        group_by(Condition_self) %>%
        summarise(
          mean_residual = mean(residual, na.rm = TRUE),
          sd_residual = sd(residual, na.rm = TRUE),
          median_residual = median(residual, na.rm = TRUE),
          n = n(),
          .groups = "drop"
        )
    )
  }
  
  # ------------------------------------------------------------
  # Save diagnostics
  # ------------------------------------------------------------
  
  diagnostics_with_nuisance[[dv]] <- df_diag
}

In [ ]:
results_lmm_with_nuisance <- list()
diagnostics_lmm_with_nuisance <- list()


# ============================================================
# Linear mixed models with nuisance covariates
# ============================================================

for (dv in dependent_variables) {
  
  cat("\n========================================\n")
  cat("LMM with nuisance covariates\n")
  cat("Type epoch:", type_epoch, "\n")
  
  if (exists("crop") && !is.null(crop)) {
    cat("Crop:", crop, "\n")
  }
  
  cat("DV:", dv, "\n")
  cat("========================================\n")
  
  if (type_epoch == "emoc") {
    
    form_lmm <- as.formula(paste(
      dv,
      "~ Condition_self * Condition_emotion +
         delta_z + theta_z + alpha_z + beta_z + gamma_z +
         (1 | Subject)"
    ))
    
  } else if (type_epoch == "self") {
    
    form_lmm <- as.formula(paste(
      dv,
      "~ Condition_self +
         delta_z + theta_z + alpha_z + beta_z + gamma_z +
         (1 | Subject)"
    ))
    
  }
  
  print(form_lmm)
  
  mod_lmm <- lmer(
    form_lmm,
    data = df_subj,
    REML = FALSE
  )
  
  cat("\n--- Model summary ---\n")
  print(summary(mod_lmm))
  
  cat("\n--- Type III ANOVA ---\n")
  print(anova(mod_lmm, type = 3))
  
  cat("\n--- Model checks: performance::check_model ---\n")
  print(check_model(mod_lmm))
  
  cat("\n--- Collinearity check ---\n")
  print(check_collinearity(mod_lmm))
  
  cat("\n--- Singularity check ---\n")
  print(check_singularity(mod_lmm))
  
  cat("\n--- Normality of residuals ---\n")
  print(check_normality(mod_lmm))
  
  cat("\n--- Heteroscedasticity check ---\n")
  print(check_heteroscedasticity(mod_lmm))
  
  # Extract residuals and fitted values
  df_diag <- df_subj %>%
    mutate(
      fitted = fitted(mod_lmm),
      residual = residuals(mod_lmm),
      scaled_residual = scale(residual)[, 1]
    )
  
  # Base R residual plots
  par(mfrow = c(2, 2))
  
  plot(
    df_diag$fitted,
    df_diag$residual,
    xlab = "Fitted values",
    ylab = "Residuals",
    main = paste("Residuals vs fitted -", dv)
  )
  abline(h = 0, lty = 2)
  
  qqnorm(
    df_diag$residual,
    main = paste("QQ plot residuals -", dv)
  )
  qqline(df_diag$residual)
  
  hist(
    df_diag$residual,
    breaks = 20,
    main = paste("Histogram residuals -", dv),
    xlab = "Residuals"
  )
  
  boxplot(
    residual ~ Subject,
    data = df_diag,
    las = 2,
    main = paste("Residuals by subject -", dv),
    ylab = "Residuals"
  )
  abline(h = 0, lty = 2)
  
  par(mfrow = c(1, 1))
  
  # Residuals by experimental condition
  if (type_epoch == "emoc") {
    
    boxplot(
      residual ~ Condition_self * Condition_emotion,
      data = df_diag,
      las = 2,
      main = paste("Residuals by condition -", dv),
      ylab = "Residuals"
    )
    abline(h = 0, lty = 2)
    
  } else if (type_epoch == "self") {
    
    boxplot(
      residual ~ Condition_self,
      data = df_diag,
      las = 2,
      main = paste("Residuals by condition -", dv),
      ylab = "Residuals"
    )
    abline(h = 0, lty = 2)
    
  }
  
  # DHARMa simulation-based residual diagnostics
  cat("\n--- DHARMa residual diagnostics ---\n")
  
  sim_res <- simulateResiduals(
    fittedModel = mod_lmm,
    n = 1000
  )
  
  plot(sim_res)
  
  print(testUniformity(sim_res))
  print(testDispersion(sim_res))
  print(testOutliers(sim_res))
  
  # Save outputs
  results_lmm_with_nuisance[[dv]] <- mod_lmm
  
  diagnostics_lmm_with_nuisance[[dv]] <- list(
    diagnostic_data = df_diag,
    dharma_residuals = sim_res,
    collinearity = check_collinearity(mod_lmm),
    singularity = check_singularity(mod_lmm),
    normality = check_normality(mod_lmm),
    heteroscedasticity = check_heteroscedasticity(mod_lmm)
  )
}

In [ ]:
# ============================================================
# Create log-transformed dataset
# ============================================================

df_subj_log <- df_subj %>%
  mutate(
    ACW_50_log = log(ACW_50),
    ACW_0_log  = log(ACW_0)
  )

# Use the log-transformed dependent variables
dependent_variables_log <- paste0(dependent_variables, "_log")


results_lmm_with_nuisance_log <- list()
diagnostics_lmm_with_nuisance_log <- list()


# ============================================================
# Linear mixed models with nuisance covariates using log-transformed DVs
# ============================================================

for (dv in dependent_variables_log) {
  
  cat("\n========================================\n")
  cat("LMM with nuisance covariates - log-transformed DV\n")
  cat("Type epoch:", type_epoch, "\n")
  
  if (exists("crop") && !is.null(crop)) {
    cat("Crop:", crop, "\n")
  }
  
  cat("DV:", dv, "\n")
  cat("========================================\n")
  
  if (type_epoch == "emoc") {
    
    form_lmm_log <- as.formula(paste(
      dv,
      "~ Condition_self * Condition_emotion +
         delta_z + theta_z + alpha_z + beta_z + gamma_z +
         (1 | Subject)"
    ))
    
  } else if (type_epoch == "self") {
    
    form_lmm_log <- as.formula(paste(
      dv,
      "~ Condition_self +
         delta_z + theta_z + alpha_z + beta_z + gamma_z +
         (1 | Subject)"
    ))
    
  }
  
  print(form_lmm_log)
  
  mod_lmm_log <- lmer(
    form_lmm_log,
    data = df_subj_log,
    REML = FALSE
  )
  
  cat("\n--- Model summary ---\n")
  print(summary(mod_lmm_log))
  
  cat("\n--- Type III ANOVA ---\n")
  print(anova(mod_lmm_log, type = 3))
  
  cat("\n--- Model checks: performance::check_model ---\n")
  print(check_model(mod_lmm_log))
  
  cat("\n--- Collinearity check ---\n")
  print(check_collinearity(mod_lmm_log))
  
  cat("\n--- Singularity check ---\n")
  print(check_singularity(mod_lmm_log))
  
  cat("\n--- Normality of residuals ---\n")
  print(check_normality(mod_lmm_log))
  
  cat("\n--- Heteroscedasticity check ---\n")
  print(check_heteroscedasticity(mod_lmm_log))
  
  # Extract residuals and fitted values
  df_diag_log <- df_subj_log %>%
    mutate(
      fitted = fitted(mod_lmm_log),
      residual = residuals(mod_lmm_log),
      scaled_residual = scale(residual)[, 1]
    )
  
  # Base R residual plots
  par(mfrow = c(2, 2))
  
  plot(
    df_diag_log$fitted,
    df_diag_log$residual,
    xlab = "Fitted values",
    ylab = "Residuals",
    main = paste("Residuals vs fitted -", dv)
  )
  abline(h = 0, lty = 2)
  
  qqnorm(
    df_diag_log$residual,
    main = paste("QQ plot residuals -", dv)
  )
  qqline(df_diag_log$residual)
  
  hist(
    df_diag_log$residual,
    breaks = 20,
    main = paste("Histogram residuals -", dv),
    xlab = "Residuals"
  )
  
  boxplot(
    residual ~ Subject,
    data = df_diag_log,
    las = 2,
    main = paste("Residuals by subject -", dv),
    ylab = "Residuals"
  )
  abline(h = 0, lty = 2)
  
  par(mfrow = c(1, 1))
  
  # Residuals by experimental condition
  if (type_epoch == "emoc") {
    
    boxplot(
      residual ~ Condition_self * Condition_emotion,
      data = df_diag_log,
      las = 2,
      main = paste("Residuals by condition -", dv),
      ylab = "Residuals"
    )
    abline(h = 0, lty = 2)
    
  } else if (type_epoch == "self") {
    
    boxplot(
      residual ~ Condition_self,
      data = df_diag_log,
      las = 2,
      main = paste("Residuals by condition -", dv),
      ylab = "Residuals"
    )
    abline(h = 0, lty = 2)
    
  }
  
  # DHARMa simulation-based residual diagnostics
  cat("\n--- DHARMa residual diagnostics ---\n")
  
  sim_res_log <- simulateResiduals(
    fittedModel = mod_lmm_log,
    n = 1000
  )
  
  plot(sim_res_log)
  
  print(testUniformity(sim_res_log))
  print(testDispersion(sim_res_log))
  print(testOutliers(sim_res_log))
  
  # Save outputs
  results_lmm_with_nuisance_log[[dv]] <- mod_lmm_log
  
  diagnostics_lmm_with_nuisance_log[[dv]] <- list(
    diagnostic_data = df_diag_log,
    dharma_residuals = sim_res_log,
    collinearity = check_collinearity(mod_lmm_log),
    singularity = check_singularity(mod_lmm_log),
    normality = check_normality(mod_lmm_log),
    heteroscedasticity = check_heteroscedasticity(mod_lmm_log)
  )
}

In [ ]:
mod <- results_lmm_with_nuisance[["ACW_0"]]

emm_emotion <- emmeans(mod, ~ Condition_emotion)

pairs(emm_emotion, adjust = "holm")

In [ ]:
# ============================================================
# NLME models with heterogeneous residual variances
# Separate Jupyter block
# ============================================================

library(nlme)
library(dplyr)
library(performance)
library(DHARMa)

results_lme_hom_log <- list()
results_lme_het_log <- list()
diagnostics_lme_het_log <- list()

# Create combined condition for heterogeneous variance by cell
if (type_epoch == "emoc") {
  df_subj_log <- df_subj_log %>%
    mutate(
      Condition_comb = interaction(
        Condition_self,
        Condition_emotion,
        drop = TRUE
      )
    )
}

for (dv in dependent_variables_log) {
  
  cat("\n========================================\n")
  cat("NLME LMM with nuisance covariates - log-transformed DV\n")
  cat("Type epoch:", type_epoch, "\n")
  
  if (exists("crop") && !is.null(crop)) {
    cat("Crop:", crop, "\n")
  }
  
  cat("DV:", dv, "\n")
  cat("========================================\n")
  
  # ------------------------------------------------------------
  # Fixed-effects formula for nlme::lme()
  # ------------------------------------------------------------
  
  if (type_epoch == "emoc") {
    
    fixed_lme_log <- as.formula(paste(
      dv,
      "~ Condition_self * Condition_emotion +
         delta_z + theta_z + alpha_z + beta_z + gamma_z"
    ))
    
  } else if (type_epoch == "self") {
    
    fixed_lme_log <- as.formula(paste(
      dv,
      "~ Condition_self +
         delta_z + theta_z + alpha_z + beta_z + gamma_z"
    ))
    
  }
  
  print(fixed_lme_log)
  
  # ------------------------------------------------------------
  # 1. Homogeneous residual variance model
  # equivalent idea to lmer model, but fitted with nlme
  # ------------------------------------------------------------
  
  cat("\n--- Homogeneous residual variance model: nlme::lme ---\n")
  
  mod_lme_hom_log <- lme(
    fixed = fixed_lme_log,
    random = ~ 1 | Subject,
    data = df_subj_log,
    method = "ML",
    na.action = na.omit,
    control = lmeControl(
      opt = "optim",
      maxIter = 100,
      msMaxIter = 100
    )
  )
  
  print(summary(mod_lme_hom_log))
  
  # ------------------------------------------------------------
  # 2. Heterogeneous residual variance model
  # ------------------------------------------------------------
  
  if (type_epoch == "emoc") {
    
    # Heterogeneous variance by full experimental cell
    # Condition_self x Condition_emotion
    weights_formula <- varIdent(form = ~ 1 | Condition_comb)
    
    cat("\n--- Heterogeneous residual variance by Condition_self x Condition_emotion ---\n")
    
  } else if (type_epoch == "self") {
    
    # Heterogeneous variance by Condition_self
    weights_formula <- varIdent(form = ~ 1 | Condition_self)
    
    cat("\n--- Heterogeneous residual variance by Condition_self ---\n")
    
  }
  
  mod_lme_het_log <- lme(
    fixed = fixed_lme_log,
    random = ~ 1 | Subject,
    weights = weights_formula,
    data = df_subj_log,
    method = "ML",
    na.action = na.omit,
    control = lmeControl(
      opt = "optim",
      maxIter = 100,
      msMaxIter = 100
    )
  )
  
  print(summary(mod_lme_het_log))
  
  # ------------------------------------------------------------
  # 3. Compare homogeneous vs heterogeneous variance models
  # ------------------------------------------------------------
  
  cat("\n--- Model comparison: homogeneous vs heterogeneous residual variance ---\n")
  print(anova(mod_lme_hom_log, mod_lme_het_log))
  
  cat("\n--- AIC comparison ---\n")
  print(AIC(mod_lme_hom_log, mod_lme_het_log))
  
  cat("\n--- BIC comparison ---\n")
  print(BIC(mod_lme_hom_log, mod_lme_het_log))
  
  # ------------------------------------------------------------
  # 4. Diagnostics for heterogeneous model
  # ------------------------------------------------------------
  
  cat("\n--- Residual diagnostics for heterogeneous model ---\n")
  
  df_diag_lme_het_log <- df_subj_log %>%
    mutate(
      fitted_lme_het = fitted(mod_lme_het_log),
      residual_lme_het = residuals(mod_lme_het_log, type = "normalized"),
      raw_residual_lme_het = residuals(mod_lme_het_log, type = "response")
    )
  
  par(mfrow = c(2, 2))
  
  plot(
    df_diag_lme_het_log$fitted_lme_het,
    df_diag_lme_het_log$residual_lme_het,
    xlab = "Fitted values",
    ylab = "Normalized residuals",
    main = paste("Normalized residuals vs fitted -", dv)
  )
  abline(h = 0, lty = 2)
  
  qqnorm(
    df_diag_lme_het_log$residual_lme_het,
    main = paste("QQ plot normalized residuals -", dv)
  )
  qqline(df_diag_lme_het_log$residual_lme_het)
  
  hist(
    df_diag_lme_het_log$residual_lme_het,
    breaks = 20,
    main = paste("Histogram normalized residuals -", dv),
    xlab = "Normalized residuals"
  )
  
  boxplot(
    residual_lme_het ~ Subject,
    data = df_diag_lme_het_log,
    las = 2,
    main = paste("Normalized residuals by subject -", dv),
    ylab = "Normalized residuals"
  )
  abline(h = 0, lty = 2)
  
  par(mfrow = c(1, 1))
  
  # ------------------------------------------------------------
  # 5. Residuals by condition
  # ------------------------------------------------------------
  
  if (type_epoch == "emoc") {
    
    boxplot(
      residual_lme_het ~ Condition_self * Condition_emotion,
      data = df_diag_lme_het_log,
      las = 2,
      main = paste("Normalized residuals by condition -", dv),
      ylab = "Normalized residuals"
    )
    abline(h = 0, lty = 2)
    
  } else if (type_epoch == "self") {
    
    boxplot(
      residual_lme_het ~ Condition_self,
      data = df_diag_lme_het_log,
      las = 2,
      main = paste("Normalized residuals by condition -", dv),
      ylab = "Normalized residuals"
    )
    abline(h = 0, lty = 2)
    
  }
  
  # ------------------------------------------------------------
  # 6. Save outputs
  # ------------------------------------------------------------
  
  results_lme_hom_log[[dv]] <- mod_lme_hom_log
  results_lme_het_log[[dv]] <- mod_lme_het_log
  
  diagnostics_lme_het_log[[dv]] <- list(
    diagnostic_data = df_diag_lme_het_log,
    homogeneous_model = mod_lme_hom_log,
    heterogeneous_model = mod_lme_het_log,
    model_comparison = anova(mod_lme_hom_log, mod_lme_het_log),
    AIC = AIC(mod_lme_hom_log, mod_lme_het_log),
    BIC = BIC(mod_lme_hom_log, mod_lme_het_log)
  )
}

# Permutations acrross ALL CHANNELS

This is wrong just to test

In [ ]:
dv <- "ACW_0"

df_subj_ch <- df %>%
  group_by(Subject, Condition_self, Condition_emotion, Channel) %>%
  summarise(
    ACW = mean(.data[[dv]], na.rm = TRUE),
    delta = mean(delta, na.rm = TRUE),
    theta = mean(theta, na.rm = TRUE),
    alpha = mean(alpha, na.rm = TRUE),
    beta  = mean(beta,  na.rm = TRUE),
    gamma = mean(gamma, na.rm = TRUE),
    n_epochs = n(),
    .groups = "drop"
  ) %>%
  group_by(Channel) %>%
  mutate(
    delta_z = as.numeric(scale(delta)),
    theta_z = as.numeric(scale(theta)),
    alpha_z = as.numeric(scale(alpha)),
    beta_z  = as.numeric(scale(beta)),
    gamma_z = as.numeric(scale(gamma))
  ) %>%
  ungroup()

In [ ]:
channels <- levels(df_subj_ch$Channel)

results_by_channel <- map_dfr(channels, function(ch) {
  
  dat_ch <- df_subj_ch %>%
    filter(Channel == ch) %>%
    droplevels()
  
  # Opcional: saltar canales con datos incompletos o insuficientes
  if (
    n_distinct(dat_ch$Subject) < 2 ||
    n_distinct(dat_ch$Condition_self) < 2 ||
    n_distinct(dat_ch$Condition_emotion) < 2
  ) {
    return(tibble(
      Channel = ch,
      Effect = NA_character_,
      F = NA_real_,
      p_parametric = NA_real_,
      p_perm = NA_real_,
      error = "insufficient data"
    ))
  }
  
  mod <- tryCatch(
    {
      aovperm(
        ACW ~ Condition_self * Condition_emotion +
          delta_z + theta_z + alpha_z + beta_z + gamma_z +
          Error(Subject / (Condition_self * Condition_emotion)),
        data = dat_ch,
        np = 5000
      )
    },
    error = function(e) e
  )
  
  if (inherits(mod, "error")) {
    return(tibble(
      Channel = ch,
      Effect = NA_character_,
      F = NA_real_,
      p_parametric = NA_real_,
      p_perm = NA_real_,
      error = mod$message
    ))
  }
  
  tab <- as.data.frame(summary(mod))
  
  tab %>%
    rownames_to_column("Effect") %>%
    filter(
      Effect %in% c(
        "Condition_self",
        "Condition_emotion",
        "Condition_self:Condition_emotion"
      )
    ) %>%
    transmute(
      Channel = ch,
      Effect,
      F = F,
      p_parametric = `parametric P(>F)`,
      p_perm = `resampled P(>F)`,
      error = NA_character_
    )
})

In [ ]:
sig_channels <- results_by_channel %>%
  filter(!is.na(p_perm), p_perm < 0.05) %>%
  arrange(Effect, p_perm)


In [ ]:
sig_self <- sig_channels %>%
  filter(Effect == "Condition_self")

sig_emotion <- sig_channels %>%
  filter(Effect == "Condition_emotion")

sig_interaction <- sig_channels %>%
  filter(Effect == "Condition_self:Condition_emotion")

sig_self
sig_emotion
sig_interaction

# 2. Permutation cluster with permuco4brain

In [ ]:
# PARAMETERS

np=5000

# cluster_forming_p=NULL


# ------------------------------------------------------------
# Define cluster-level significance threshold
# ------------------------------------------------------------

cluster_alpha <- 0.05


### Average and scale of data

In [ ]:
df_chan <- df %>%
  group_by(Subject, Condition_self, Condition_emotion, Channel) %>%
  summarise(
    ACW_0  = mean(ACW_0, na.rm = TRUE),
    ACW_50 = mean(ACW_50, na.rm = TRUE),
    delta = mean(delta, na.rm = TRUE),
    theta = mean(theta, na.rm = TRUE),
    alpha = mean(alpha, na.rm = TRUE),
    beta  = mean(beta,  na.rm = TRUE),
    gamma = mean(gamma, na.rm = TRUE),
    n_epochs = n(),
    .groups = "drop"
  )

In [ ]:
df_chan <- df_chan %>%
  mutate(
    delta_z = as.numeric(scale(delta)),
    theta_z = as.numeric(scale(theta)),
    alpha_z = as.numeric(scale(alpha)),
    beta_z  = as.numeric(scale(beta)),
    gamma_z = as.numeric(scale(gamma))
  )

In [ ]:
# ------------------------------------------------------------
# 2. Keep only channels present in the graph and set order
# ------------------------------------------------------------

graph_channels_names <- V(graph_channels)$name

df_chan <- df_chan %>%
  filter(Channel %in% graph_channels_names)

# Sanity checks
cat("N channels in graph:", length(graph_channels_names), "\n")
cat("N channels in df_chan:", length(unique(df_chan$Channel)), "\n")

print(setdiff(unique(df_chan$Channel), graph_channels_names))
print(setdiff(graph_channels_names, unique(df_chan$Channel)))

In [ ]:
# ------------------------------------------------------------
# 3. Build design table: one row per
#    Subject × Condition_self × Condition_emotion
#    Nuisance covariates are averaged across channels
# ------------------------------------------------------------


design <- df_chan %>%
  group_by(Subject, Condition_self, Condition_emotion) %>%
  summarise(
    delta = mean(delta, na.rm = TRUE),
    theta = mean(theta, na.rm = TRUE),
    alpha = mean(alpha, na.rm = TRUE),
    beta  = mean(beta,  na.rm = TRUE),
    gamma = mean(gamma, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(
    Subject = factor(Subject),
    Condition_self = factor(Condition_self),
    Condition_emotion = factor(Condition_emotion),
    delta_z = as.numeric(scale(delta)),
    theta_z = as.numeric(scale(theta)),
    alpha_z = as.numeric(scale(alpha)),
    beta_z  = as.numeric(scale(beta)),
    gamma_z = as.numeric(scale(gamma))
  ) %>%
  arrange(Subject, Condition_self, Condition_emotion)

cat("N design rows:", nrow(design), "\n")

# Check that there is only one row per Subject × Condition.
# Expected output: an empty tibble, e.g. "# A tibble: 0 × 3".
# If rows appear here, the design table still contains duplicated
# Subject × Condition observations.
design_check <- design %>%
  count(Subject, Condition_self, Condition_emotion) %>%
  filter(n > 1)

print(design_check)

stopifnot(nrow(design_check) == 0)

In [ ]:
# ------------------------------------------------------------
# Cluster-forming threshold
# ------------------------------------------------------------

if (is.null(cluster_forming_p)) {
  
  cluster_forming_F <- NULL
  
  cat("cluster_forming_p is NULL: using brainperm default threshold\n")
  
} else {
  
  # Approximate denominator df for F threshold.
  # This is used only to define the cluster-forming threshold.
  df1 <- 1
  df2 <- nrow(design) - length(coef(lm(
    rep(1, nrow(design)) ~
      Condition_self * Condition_emotion +
      delta_z + theta_z + alpha_z + beta_z + gamma_z,
    data = design
  )))
  
  cluster_forming_F <- qf(
    p = 1 - cluster_forming_p,
    df1 = df1,
    df2 = df2
  )
  
  cat("cluster_forming_p:", cluster_forming_p, "\n")
  cat("cluster_forming_F:", cluster_forming_F, "\n")
}

In [ ]:
make_signal_array <- function(df_chan, design, dv, channel_order) {
  
  df_wide <- df_chan %>%
    select(
      Subject,
      Condition_self,
      Condition_emotion,
      Channel,
      value = all_of(dv)
    ) %>%
    mutate(
      Subject = factor(
        Subject,
        levels = levels(design$Subject)
      ),
      Condition_self = factor(
        Condition_self,
        levels = levels(design$Condition_self)
      ),
      Condition_emotion = factor(
        Condition_emotion,
        levels = levels(design$Condition_emotion)
      ),
      Channel = factor(
        Channel,
        levels = channel_order
      )
    ) %>%
    arrange(
      Subject,
      Condition_self,
      Condition_emotion,
      Channel
    ) %>%
    pivot_wider(
      names_from = Channel,
      values_from = value
    )
  
  # Align rows with design
  df_wide <- design %>%
    select(
      Subject,
      Condition_self,
      Condition_emotion
    ) %>%
    left_join(
      df_wide,
      by = c(
        "Subject",
        "Condition_self",
        "Condition_emotion"
      )
    )
  
  mat <- as.matrix(df_wide[, channel_order])
  
  signal <- array(
    mat,
    dim = c(nrow(mat), 1, length(channel_order)),
    dimnames = list(
      observation = paste(
        design$Subject,
        design$Condition_self,
        design$Condition_emotion,
        sep = "_"
      ),
      sample = "spatial_only",
      channel = channel_order
    )
  )
  
  return(signal)
}

In [ ]:
# ------------------------------------------------------------
# Loop over dependent variables
# ------------------------------------------------------------


for (dv in dependent_variables) {
  
  cat("\n============================================================\n")
  cat("Running analysis for dependent variable:", dv, "\n")
  cat("============================================================\n")

  signal_acw <- make_signal_array(
    df_chan = df_chan,
    design = design,
    dv = dv,
    channel_order = graph_channels_names
  )
  dim(signal_acw)

  stopifnot(dim(signal_acw)[1] == nrow(design))
  stopifnot(dim(signal_acw)[2] == 1)
  stopifnot(identical(dimnames(signal_acw)[[3]], V(graph_channels)$name))

  cat("✅ signal_acw dimensions are OK\n")
  df_chan_check <- df_chan %>%
    count(Subject, Condition_self, Condition_emotion, Channel) %>%
    filter(n > 1)

  print(df_chan_check)
  stopifnot(nrow(df_chan_check) == 0)
  # ------------------------------------------------------------
  # 5. Spatial-only cluster permutation with permuco4brain
  # ------------------------------------------------------------

  # ------------------------------------------------------------
  # Use sequential execution first

  # ------------------------------------------------------------
  # Parallel backend
  # ------------------------------------------------------------
  # Use a moderate number of workers first.
  # On Windows, each worker may receive a copy of large objects.

  plan(multisession, workers = 4)

  # ------------------------------------------------------------
  # Increase maximum size of objects exported to workers
  # ------------------------------------------------------------
  # 32 GB is reasonable with 120 GB RAM.
  # Increase to 64 GB only if needed.

  options(future.globals.maxSize = 32 * 1024^3)



  formula_acw <- signal_acw ~ 
    Condition_self * Condition_emotion +
    delta_z + theta_z + alpha_z + beta_z + gamma_z +
    Error(Subject / (Condition_self * Condition_emotion))

  effects_to_test <- data.frame(
    effect_index = c(1, 2, 8),
    effect_name = c(
      "Condition_self",
      "Condition_emotion",
      "Condition_self:Condition_emotion"
    ),
    stringsAsFactors = FALSE
  )
  for (i in seq_len(nrow(effects_to_test))) {
    
    effect_index <- effects_to_test$effect_index[i]
    effect_name  <- effects_to_test$effect_name[i]
    
    cat("\n------------------------------------------------------------\n")
    cat("Testing effect:", effect_name, "\n")
    cat("Effect index:", effect_index, "\n")
    cat("------------------------------------------------------------\n")
    
    set.seed(123)
    
    model_acw <- brainperm(
      formula = formula_acw,
      data = design,
      graph = graph_channels,
      np = np,
      multcomp = "clustermass",
      test = "fisher", 
      effect = effect_index
    )

    # # model_acw
    # # summary(model_acw)
    ## Information of the model
    class(model_acw)
    names(model_acw)
    str(model_acw, max.level = 2)
    # Extract the multiple-comparison results for the effect of interest.
    # Here, effect = Condition, because brainperm was run with effect = 1.
    condition_mc <- model_acw$multiple_comparison[[effect_name]]

    # Inspect the available components for the Condition effect.
    # Typically includes uncorrected channel-wise results and cluster-level results.
    names(condition_mc)

    # Inspect the structure of the Condition results without printing everything.
    # Useful to confirm where statistics, p-values, clusters, and thresholds are stored.
    str(condition_mc, max.level = 2)

    # Extract the cluster object for the clustermass correction.
    # This contains the spatial clusters formed from suprathreshold channels.
    cluster_obj <- condition_mc$clustermass$cluster

    # Cluster membership for each suprathreshold node/channel.
    # Values indicate which cluster each channel/node belongs to.
    membership <- cluster_obj$membership

    # Number of channels/nodes included in each cluster.
    cluster_sizes <- cluster_obj$csize

    # Cluster mass for each cluster.
    # Usually the sum of the channel-wise statistics inside the cluster.
    cluster_masses <- cluster_obj$clustermass

    # Cluster-level corrected p-values.
    # These are the p-values used to decide which clusters survive correction.
    cluster_pvalues <- cluster_obj$pvalue
    # ------------------------------------------------------------
    # Create cluster summary table
    # ------------------------------------------------------------

    # ------------------------------------------------------------
    # Create cluster summary table safely
    # ------------------------------------------------------------

    n_clusters <- length(cluster_sizes)

    if (n_clusters > 0) {
      
      clusters_table <- data.frame(
        dv = rep(dv, n_clusters),
        effect = rep(effect_name, n_clusters),
        cluster_id = seq_len(n_clusters),
        n_channels = as.integer(cluster_sizes),
        clustermass = as.numeric(cluster_masses),
        pvalue = as.numeric(cluster_pvalues),
        stringsAsFactors = FALSE
      )
      
    } else {
      
      clusters_table <- data.frame(
        dv = character(),
        effect = character(),
        cluster_id = integer(),
        n_channels = integer(),
        clustermass = numeric(),
        pvalue = numeric(),
        stringsAsFactors = FALSE
      )
    }

    print(clusters_table)


    # ------------------------------------------------------------
    # Keep only clusters surviving cluster-level correction
    # ------------------------------------------------------------

    significant_clusters_table <- clusters_table %>%
      filter(pvalue < cluster_alpha)


    # ------------------------------------------------------------
    # Extract channel names only for significant clusters
    # ------------------------------------------------------------

    if (nrow(significant_clusters_table) > 0) {
      
      significant_cluster_channels_table <- lapply(
        significant_clusters_table$cluster_id,
        function(cl_id) {
          
          channels <- names(membership)[membership == cl_id]
          channels <- sub("_[0-9]+$", "", channels)
          channels <- unique(channels)
          
          data.frame(
            dv = dv,
            effect = effect_name,
            cluster_id = cl_id,
            Channel = channels,
            stringsAsFactors = FALSE
          )
        }
      ) %>%
        bind_rows()
      
    } else {
      
      significant_cluster_channels_table <- data.frame(
        dv = character(),
        effect = character(),
        cluster_id = integer(),
        Channel = character(),
        stringsAsFactors = FALSE
      )
    }

    # ------------------------------------------------------------
    # Add channel list to significant cluster summary table
    # ------------------------------------------------------------

    if (nrow(significant_cluster_channels_table) > 0) {
      
      significant_clusters_summary <- significant_cluster_channels_table %>%
        group_by(dv, effect, cluster_id) %>%
        summarise(
          channels = paste(Channel, collapse = ", "),
          .groups = "drop"
        ) %>%
        right_join(
          significant_clusters_table,
          by = c("dv", "effect", "cluster_id")
        ) %>%
        select(
          dv,
          effect,
          cluster_id,
          n_channels,
          clustermass,
          pvalue,
          channels
        )
      
    } else {
      
      significant_clusters_summary <- significant_clusters_table %>%
        mutate(channels = character()) %>%
        select(
          dv,
          effect,
          cluster_id,
          n_channels,
          clustermass,
          pvalue,
          channels
        )
    }

    print(significant_clusters_summary)
    print(significant_cluster_channels_table)

    # ------------------------------------------------------------
    # Save significant cluster channels and summary
    # ------------------------------------------------------------

    effect_safe <- gsub(":", "_x_", effect_name)

    output_suffix <- paste0(
      "_",
      dv,
      "_",
      effect_safe,
      "_",
      layer_script,
      if (dynamic) "_dynamic" else "",
      "_",
      type_epoch
    )

    channels_file <- file.path(
      statistical_models,
      paste0("significant_cluster_channels", output_suffix, ".csv")
    )

    summary_file <- file.path(
      statistical_models,
      paste0("significant_clusters_summary", output_suffix, ".csv")
    )

    write.csv(
      significant_cluster_channels_table,
      file = channels_file,
      row.names = FALSE
    )

    write.csv(
      significant_clusters_summary,
      file = summary_file,
      row.names = FALSE
    )

    cat("Saved significant cluster channels to:\n", channels_file, "\n")
    cat("Saved significant cluster summary to:\n", summary_file, "\n")

  }  # closes effect_name loop
}    # closes dv loop


In [ ]:
statistical_models

In [ ]:
cat("Effect:", effect_name, "\n")
cat("cluster_forming_F:", cluster_forming_F, "\n")
cat("is.na(cluster_forming_F):", is.na(cluster_forming_F), "\n")

cat("Any NA in signal_acw?:", anyNA(signal_acw), "\n")
cat("N NA in signal_acw:", sum(is.na(signal_acw)), "\n")

cat("Any NA in design?:", anyNA(design), "\n")
print(colSums(is.na(design)))

cat("Any non-finite in signal_acw?:", any(!is.finite(signal_acw)), "\n")
cat("N non-finite in signal_acw:", sum(!is.finite(signal_acw)), "\n")

num_cols <- sapply(design, is.numeric)
print(sapply(design[, num_cols, drop = FALSE], function(x) {
  c(
    any_na = anyNA(x),
    any_nonfinite = any(!is.finite(x)),
    sd = sd(x, na.rm = TRUE)
  )
}))

# end 




# alternatives for cluster permutation




In [ ]:
# ------------------------------------------------------------
# NOTE ON CHANNEL-SPECIFIC NUISANCE COVARIATES
# ------------------------------------------------------------
#
# In permuco4brain::brainperm(), the dependent variable is provided as a
# 3D signal array with dimensions:
#
#   design/observations × samples/time × channels/space
#
# The data argument contains the experimental design, with one row per
# observation/design row. The channel dimension is represented in the signal
# array and in the spatial adjacency graph. Therefore, covariates included
# in the formula via data are observation-level covariates, not
# channel-specific covariates.
#
# This means that:
#
#   signal ~ Condition + delta_z + theta_z + alpha_z + ...
#
# represents:
#
#   ACW[channel] ~ Condition + observation-level power covariates
#
# and does not directly represent:
#
#   ACW[channel] ~ Condition + power[channel]
#
# References for this data structure:
# https://jaromilfrossard.github.io/permuco4brain/reference/brainperm.html
# https://github.com/jaromilfrossard/permuco4brain
# https://www.jstatsoft.org/article/view/v099i15
#
#
# ------------------------------------------------------------
# Strategy A: channel-wise residualization + brainperm
# ------------------------------------------------------------
#
# First, ACW is residualized separately at each channel with respect to
# channel-specific oscillatory power:
#
#   ACW_channel ~ delta_channel + theta_channel + alpha_channel
#                 + beta_channel + gamma_channel
#
# The residual ACW maps are then submitted to brainperm to test the effect
# of Condition with spatial cluster correction:
#
#   residual_ACW[channel] ~ Condition + Error(Subject / Condition)
#
# This keeps the spatial cluster-permutation framework from permuco4brain,
# while controlling channel-wise spectral power before the permutation test.
# However, this should be described as a sensitivity analysis, because the
# nuisance regression is performed before the cluster-permutation model,
# rather than as a full Freedman-Lane permutation of the complete model.
#
# References for Strategy A:
# https://brainder.org/2020/05/19/simplifying-freedman-lane/
# https://edepot.wur.nl/536311
# https://jaromilfrossard.github.io/permuco4brain/reference/brainperm.html
#
#
# ------------------------------------------------------------
# Strategy B: custom mass-univariate cluster permutation
# ------------------------------------------------------------
#
# A more exact approach is to implement a custom permutation procedure.
# For each permutation and for each channel, fit the full channel-wise model:
#
#   ACW_channel ~ Condition + delta_channel + theta_channel + alpha_channel
#                 + beta_channel + gamma_channel
#
# Extract the test statistic for Condition at each channel, form spatial
# clusters using the channel adjacency graph, and build the null distribution
# from the maximum cluster mass across permutations.
#
# This directly tests the effect of Condition while controlling for
# channel-specific spectral power. It is statistically closer to the desired
# model, but requires a custom implementation and careful handling of the
# within-subject permutation scheme.
#
# References for Strategy B:
# https://mne.tools/stable/generated/mne.stats.permutation_cluster_test.html
# https://mne.discourse.group/t/spatio-temporal-custer-permutation-with-linear-regression/9073
# https://www.fieldtriptoolbox.org/tutorial/stats/cluster_permutation_freq/
# https://pmc.ncbi.nlm.nih.gov/articles/PMC4510917/

https://chatgpt.com/c/6a3e4cae-e0c4-83ed-86b8-de6c913d1025

For serialization